# 01 — Chargement de la base FINESS

Charge en parallèle les **EG** (établissements géographiques) et les **EJ** (entités juridiques) depuis SQL Server, en filtrant sur les structures actives.

Sortie : 2 fichiers parquet dans `data/interim/`.

In [1]:
# Installation des packages
%pip install -r ../requirements.txt --quiet

Note: you may need to restart the kernel to use updated packages.


In [2]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
from src.connexion    import get_finess_connection
from src.display      import afficher_tableau
from config.settings  import (
    FINESS_EG_RAW, FINESS_EJ_RAW, INTERIM_DIR,
)

INTERIM_DIR.mkdir(parents=True, exist_ok=True)

## 1. EG (établissements géographiques)

In [3]:
conn = get_finess_connection()

query_eg = """
    SELECT idstructure_stru, nmfinessej_stru, nmfinessetab_stru,
           categetab_stru, nmsiret_stru, raisonsociale_stru,
           nmvoie_stru, lbtypevoie_stru, lbvoie_stru, cdcommune_stru
    FROM BICOEUR_DWH_SNAPSHOT.dbo.dwh_structure
    WHERE topsource_stru = 'FINESS'
      AND typeidpm_stru  = 'EG'
      AND (dtfermestruct_stru IS NULL OR dtfermestruct_stru >= SYSDATETIME())
"""
df_eg = pd.read_sql(query_eg, conn)
print(f'EG FINESS actifs : {len(df_eg):,}')
print(f'  avec SIRET     : {df_eg["nmsiret_stru"].notna().sum():,}')

df_eg.to_parquet(FINESS_EG_RAW, index=False)
print(f'Sauvegardé : {FINESS_EG_RAW}')

afficher_tableau(df_eg, 'Aperçu EG FINESS')

/tmp/ipykernel_752/3453939771.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_eg = pd.read_sql(query_eg, conn)


EG FINESS actifs : 104,612
  avec SIRET     : 91,483
Sauvegardé : /home/jovyan/work/projet_finess_sirene/data/interim/finess_eg.parquet


idstructure_stru,nmfinessej_stru,nmfinessetab_stru,categetab_stru,nmsiret_stru,raisonsociale_stru,nmvoie_stru,lbtypevoie_stru,lbvoie_stru,cdcommune_stru
1931885,240011197,240011205,620,51494491700012,PHARMACIE DE VESONE,81,R,CLAUDE BERNARD,24322
1931887,240011247,240011254,620,83899132100012,PHARMACIE FENELON,25,CRS,FENELON,24322
1931888,240011262,240011270,620,43352756100016,PHARMACIE REYDY,5,PL,FRANCHEVILLE,24322
1931891,240011312,240011320,620,90056938500017,PHARMACIE CHAPARD,36,R,Gambetta,24322
1931893,240011346,240011353,620,32784546700015,PHARMACIE DAUMARES,104,R,GAMBETTA,24322


## 2. EJ (entités juridiques)

In [4]:
query_ej = """
    SELECT idstructure_stru, nmfinessej_stru, nmfinessetab_stru,
           categetab_stru, nmsiren_stru, nmsiret_stru,
           raisonsociale_stru, nmvoie_stru, lbtypevoie_stru, 
           lbvoie_stru, cdcommune_stru, dtouvertstruct_stru, cdape_stru
    FROM BICOEUR_DWH_SNAPSHOT.dbo.dwh_structure
    WHERE topsource_stru = 'FINESS'
      AND typeidpm_stru  = 'EJ'
      AND (dtfermestruct_stru IS NULL OR dtfermestruct_stru >= SYSDATETIME())
"""
df_ej = pd.read_sql(query_ej, conn)
conn.close()

print(f'EJ FINESS actifs : {len(df_ej):,}')
print(f'  avec SIREN     : {df_ej["nmsiren_stru"].notna().sum():,}')
print(f'  avec APE       : {df_ej["cdape_stru"].notna().sum():,}')
print(f'  avec date ouv. : {df_ej["dtouvertstruct_stru"].notna().sum():,}')

df_ej.to_parquet(FINESS_EJ_RAW, index=False)
print(f'Sauvegardé : {FINESS_EJ_RAW}')

afficher_tableau(df_ej, 'Aperçu EJ FINESS')

/tmp/ipykernel_752/4229738088.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_ej = pd.read_sql(query_ej, conn)


EJ FINESS actifs : 54,097
  avec SIREN     : 51,999
  avec APE       : 20,309
  avec date ouv. : 54,075
Sauvegardé : /home/jovyan/work/projet_finess_sirene/data/interim/finess_ej.parquet


idstructure_stru,nmfinessej_stru,nmfinessetab_stru,categetab_stru,nmsiren_stru,nmsiret_stru,raisonsociale_stru,nmvoie_stru,lbtypevoie_stru,lbvoie_stru,cdcommune_stru,dtouvertstruct_stru,cdape_stru
1843328,300010543,None,None,263003873,None,CCAS BARJAC,None,None,None,30029,2001-01-01 00:00:00,8899B
1843331,300010709,None,None,None,None,AAD SOLEIL,None,None,None,30217,2006-04-06 00:00:00,None
1843334,300010808,None,None,424466183,None,COLLECTIF ASSOCIATIF DU BASSIN ALESIEN,55,GR,JEAN MOULIN,30007,2006-07-12 00:00:00,8810B
1843335,300010816,None,None,494422058,None,PHARMACIE CHAINIEUX,1601,RTE,DE NIMES,30259,2017-10-16 00:00:00,None
1843336,300010840,None,None,953175106,None,PHARMACIE DE SAINT GERVASY,1,PL,du Marche,30257,1997-07-11 00:00:00,None
